# Cartes d'identité des fichiers CSV et SQL

Pour chaque CSV et script SQL de `datas/`, on produit une carte d'identité markdown dans
`artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md`.

In [5]:
import csv
from datetime import datetime
from pathlib import Path

import pandas as pd
import sqlglot
from sqlglot import exp

DATA_DIR = Path("datas")
OUT_DIR = Path("artifacts/ingestion/identities")

## Fonctions

In [6]:
def detect_charset(path):
    """Encodage du fichier : UTF-8 (avec ou sans BOM), sinon Windows-1252, sinon Latin-1."""
    raw = Path(path).read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    for enc in ("utf-8", "cp1252"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            pass
    return "latin-1"  # décode toujours


def detect_separator(path, charset, n_lines=20):
    """Séparateur deviné sur les premières lignes (, ; tabulation |)."""
    with open(path, encoding=charset, newline="") as f:
        sample = "".join(f.readline() for _ in range(n_lines))
    return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter


def human_size(n_bytes):
    """9999.99 Octets | Ko | Mo | Go (base 1024)."""
    size = float(n_bytes)
    for unit in ("Octets", "Ko", "Mo"):
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} Go"


DATE_FORMATS = [("DATE", "%Y-%m-%d"), ("DATETIME", "%Y-%m-%d %H:%M:%S"), ("TIME", "%H:%M"), ("TIME", "%H:%M:%S")]


def infer_type(name, series):
    """IDENTITY, BOOLEAN, INTEGER, FLOAT, DATE, DATETIME, TIME ou TEXT."""
    if name == "id" or name.endswith("_id"):
        return "IDENTITY"
    values = series.dropna()
    if pd.api.types.is_numeric_dtype(series):
        if len(values) and values.isin([0, 1]).all():
            return "BOOLEAN"
        return "INTEGER" if pd.api.types.is_integer_dtype(series) or (values % 1 == 0).all() else "FLOAT"
    for type_name, fmt in DATE_FORMATS:
        if len(values) and pd.to_datetime(values, format=fmt, errors="coerce").notna().all():
            return type_name
    return "TEXT"


def column_stats(series, col_type):
    """Min / Max (+ Mean / Median pour les numériques)."""
    if col_type in ("INTEGER", "FLOAT"):
        return {"min": series.min(), "max": series.max(), "mean": round(series.mean(), 3), "median": series.median()}
    # TIME au format HH:MM(:SS) : l'ordre alphabétique est l'ordre chronologique
    values = series.dropna() if col_type == "TIME" else pd.to_datetime(series)
    if col_type == "DATE":
        values = values.dt.date
    return {"min": values.min(), "max": values.max(), "mean": "", "median": ""}


STAT_TYPES = ("INTEGER", "FLOAT", "DATE", "DATETIME", "TIME")


def render_header(path, begin, end, charset):
    """En-tête commun : source, taille, horaires, encodage."""
    return [
        f"# Carte d'identité : {path.name}",
        "",
        f"- Source: {path}",
        f"- Filesize: {human_size(path.stat().st_size)}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {charset.upper()}",
    ]


def render_table(df, types, level="##"):
    """Lignes/colonnes, liste des colonnes typées et stats des colonnes numériques et dates."""
    # une colonne déclarée sans aucune valeur (ex. DEFAULT NOW() en SQL) n'a pas de stats
    stats = {col: column_stats(df[col], t) for col, t in types.items()
             if t in STAT_TYPES and col in df.columns and df[col].notna().any()}
    return [
        f"- Rows Number: {len(df)}",
        f"- Cols Number: {len(types)}",
        "",
        f"{level} Column list",
        "",
        "| Colname | Infered Type |",
        "| ------- | ------------ |",
        *[f"| {col} | {t} |" for col, t in types.items()],
        "",
        f"{level} Numeric and date columns",
        "",
        "| Colname | Min value | Max value | Mean value | Median value |",
        "| ------- | --------- | --------- | ---------- | ------------ |",
        *[f"| {col} | {s['min']} | {s['max']} | {s['mean']} | {s['median']} |" for col, s in stats.items()],
        "",
    ]


def build_identity_card(path):
    """Lit le CSV et renvoie (carte markdown, heure de début)."""
    path = Path(path).resolve()
    begin = datetime.now()
    charset = detect_charset(path)
    sep = detect_separator(path, charset)
    df = pd.read_csv(path, sep=sep, encoding=charset)
    types = {col: infer_type(col, df[col]) for col in df.columns}
    body = render_table(df, types)
    end = datetime.now()
    return "\n".join(render_header(path, begin, end, charset) + body), begin


# Types SQL déclarés -> nos types (le reste : VARCHAR, TEXT... -> TEXT)
SQL_TYPES = {
    "INT": "INTEGER", "SMALLINT": "INTEGER", "BIGINT": "INTEGER",
    "DECIMAL": "FLOAT", "FLOAT": "FLOAT", "DOUBLE": "FLOAT",
    "DATE": "DATE", "TIMESTAMP": "DATETIME", "TIMESTAMPTZ": "DATETIME", "TIME": "TIME",
    "BOOLEAN": "BOOLEAN",
}


def sql_column_type(col):
    """Type d'une colonne de CREATE TABLE : IDENTITY pour clé primaire/étrangère ou *_id, sinon type déclaré."""
    is_key = any(isinstance(c.kind, (exp.PrimaryKeyColumnConstraint, exp.Reference)) for c in col.constraints)
    if is_key or col.name == "id" or col.name.endswith("_id"):
        return "IDENTITY"
    return SQL_TYPES.get(col.kind.this.name, "TEXT")


def parse_sql_tables(sql_text):
    """{table: (DataFrame des INSERT, {colonne: type})} à partir d'un script PostgreSQL (lu, jamais exécuté)."""
    types, rows = {}, {}
    for stmt in sqlglot.parse(sql_text, read="postgres"):
        if isinstance(stmt, exp.Create) and stmt.kind == "TABLE":
            name = stmt.this.this.name
            types[name] = {c.name: sql_column_type(c) for c in stmt.this.expressions if isinstance(c, exp.ColumnDef)}
            rows[name] = []
        elif isinstance(stmt, exp.Insert):
            name = stmt.this.this.name
            cols = [c.name for c in stmt.this.expressions]
            rows[name] += [dict(zip(cols, (v.to_py() for v in row.expressions))) for row in stmt.expression.expressions]
    return {name: (pd.DataFrame(rows[name]), types[name]) for name in types}


def build_sql_identity_card(path):
    """Lit le script SQL et renvoie (carte markdown avec une section par table, heure de début)."""
    path = Path(path).resolve()
    begin = datetime.now()
    charset = detect_charset(path)
    body = []
    for name, (df, types) in parse_sql_tables(path.read_text(encoding=charset)).items():
        body += ["", f"## Table : {name}", ""] + render_table(df, types, level="###")
    end = datetime.now()
    return "\n".join(render_header(path, begin, end, charset) + body), begin


def write_card(path):
    """Écrit la carte dans artifacts/ingestion/identities/AAAAMMJJ/<nom>_HHMMSS.md."""
    path = Path(path)
    build = build_sql_identity_card if ".sql" in path.suffixes else build_identity_card
    card, begin = build(path)
    out = OUT_DIR / f"{begin:%Y%m%d}" / f"{path.name.split('.')[0]}_{begin:%H%M%S}.md"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(card, encoding="utf-8")
    return out


## Auto-contrôle des fonctions

In [7]:
assert human_size(500) == "500.00 Octets"
assert human_size(2048) == "2.00 Ko"
assert human_size(8323471) == "7.94 Mo"
assert infer_type("machine_id", pd.Series(["MACH-01", "MACH-01"])) == "IDENTITY"
assert infer_type("flag", pd.Series([0, 1, 1])) == "BOOLEAN"
assert infer_type("n", pd.Series([3, 5])) == "INTEGER"
assert infer_type("t", pd.Series([1.5, 2.0])) == "FLOAT"
assert infer_type("d", pd.Series(["2025-06-01", "2025-06-02"])) == "DATE"
assert infer_type("ts", pd.Series(["2025-06-01 00:00:00"])) == "DATETIME"
assert infer_type("h", pd.Series(["05:42", "21:08"])) == "TIME"
assert infer_type("c", pd.Series(["abc", "2025-06-01"])) == "TEXT"
mini_sql = """
CREATE TABLE t (code VARCHAR(8) PRIMARY KEY, d DATE, v NUMERIC(4,2), n INTEGER, ok BOOLEAN DEFAULT TRUE);
INSERT INTO t (code, d, v, n) VALUES ('A', '2025-01-01', 1.5, 3), ('B', NULL, 2.5, 4);
"""
df_t, types_t = parse_sql_tables(mini_sql)["t"]
assert types_t == {"code": "IDENTITY", "d": "DATE", "v": "FLOAT", "n": "INTEGER", "ok": "BOOLEAN"}
assert len(df_t) == 2 and list(df_t.columns) == ["code", "d", "v", "n"]
assert df_t["v"].mean() == 2.0 and df_t["d"].isna().sum() == 1
print("OK")

OK


## Génération des cartes

In [8]:
for data_file in sorted(DATA_DIR.glob("*.csv")) + sorted(DATA_DIR.glob("*.sql*")):
    print(write_card(data_file))

artifacts/ingestion/identities/20261120/releves_incidents_151346.md
artifacts/ingestion/identities/20261120/telemetry_151346.md
